<a href="https://colab.research.google.com/github/sajina464/C-Programs/blob/main/day_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

financial_evidence = [
    {
        "id": "E1",
        "topic": "Credit score",
        "source": "credit_score_guide",
        "text": (
            "Paying credit card bills and loan EMIs on time can help "
            "maintain a positive credit history. Late payments may "
            "negatively affect credit scores."
        )
    },
    {
        "id": "E2",
        "topic": "Credit utilization",
        "source": "credit_utilization_guide",
        "text": (
            "Keeping credit utilization low is generally preferable. "
            "Around 30 percent or less is a commonly used guideline, "
            "but it does not guarantee a higher credit score."
        )
    },
    {
        "id": "E3",
        "topic": "Loan eligibility",
        "source": "loan_eligibility_guide",
        "text": (
            "Loan approval depends on lender requirements, income, "
            "existing debts, credit history and repayment capacity. "
            "A credit score alone cannot guarantee approval."
        )
    },
    {
        "id": "E4",
        "topic": "Debt-to-income ratio",
        "source": "debt_management_guide",
        "text": (
            "Debt-to-income ratio is calculated by dividing total "
            "monthly debt payments by gross monthly income and "
            "multiplying by 100."
        )
    },
    {
        "id": "E5",
        "topic": "Budgeting",
        "source": "budgeting_guide",
        "text": (
            "Create a monthly budget, track spending and prioritize "
            "essential expenses. An emergency fund can help cover "
            "unexpected expenses."
        )
    },
    {
        "id": "E6",
        "topic": "Credit report errors",
        "source": "credit_report_guide",
        "text": (
            "Review credit reports for inaccurate accounts or payment "
            "records. Contact the relevant credit bureau or lender "
            "and provide supporting documentation to dispute errors."
        )
    }
]

print("Evidence documents loaded:", len(financial_evidence))


Evidence documents loaded: 6


In [3]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

evidence_texts = [item["text"] for item in financial_evidence]

evidence_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

evidence_matrix = evidence_vectorizer.fit_transform(evidence_texts)


def find_evidence(question, top_k=2, threshold=0.10):
    question_vector = evidence_vectorizer.transform([question])

    scores = cosine_similarity(
        question_vector, evidence_matrix
    )[0]

    ranked = scores.argsort()[::-1]
    matches = []

    for index in ranked:
        if scores[index] < threshold:
            break

        item = financial_evidence[index]

        matches.append({
            "id": item["id"],
            "topic": item["topic"],
            "source": item["source"],
            "text": item["text"],
            "similarity": round(float(scores[index]), 3)
        })

        if len(matches) >= top_k:
            break

    return matches



def explain_financial_query(question):
    evidence = find_evidence(question, top_k=3)

    if not evidence:
        return {
            "question": question,
            "explanation": (
                "Insufficient supporting evidence in the available "
                "financial documents. Please consult a reliable source."
            ),
            "evidence_ids": [],
            "sources": [],
            "evidence_status": "Insufficient evidence"
        }

    explanation_parts = []

    for item in evidence:
        explanation_parts.append(
            f"{item['text']} [Evidence: {item['id']}]"
        )

    explanation = " ".join(explanation_parts)

    return {
        "question": question,
        "explanation": explanation,
        "evidence_ids": [item["id"] for item in evidence],
        "sources": [item["source"] for item in evidence],
        "evidence_status": "Evidence retrieved; verify relevance"
    }


# Example test
result = explain_financial_query(
    "How can I improve my credit score?"
)

print("QUESTION:", result["question"])
print("\nEXPLANATION:", result["explanation"])
print("\nEVIDENCE IDS:", result["evidence_ids"])
print("SOURCES:", result["sources"])
print("STATUS:", result["evidence_status"])

question = "How can I improve my credit score?"

for evidence in find_evidence(question):
    print("Evidence ID:", evidence["id"])
    print("Source:", evidence["source"])
    print("Evidence:", evidence["text"])
    print()


QUESTION: How can I improve my credit score?

EXPLANATION: Keeping credit utilization low is generally preferable. Around 30 percent or less is a commonly used guideline, but it does not guarantee a higher credit score. [Evidence: E2] Loan approval depends on lender requirements, income, existing debts, credit history and repayment capacity. A credit score alone cannot guarantee approval. [Evidence: E3] Paying credit card bills and loan EMIs on time can help maintain a positive credit history. Late payments may negatively affect credit scores. [Evidence: E1]

EVIDENCE IDS: ['E2', 'E3', 'E1']
SOURCES: ['credit_utilization_guide', 'loan_eligibility_guide', 'credit_score_guide']
STATUS: Evidence retrieved; verify relevance
Evidence ID: E2
Source: credit_utilization_guide
Evidence: Keeping credit utilization low is generally preferable. Around 30 percent or less is a commonly used guideline, but it does not guarantee a higher credit score.

Evidence ID: E3
Source: loan_eligibility_guide
Ev

In [5]:

import pandas as pd

test_questions = [
    "How can I improve my credit score?",
    "What is credit utilization?",
    "Does a high credit score guarantee loan approval?",
    "How do I calculate my debt-to-income ratio?",
    "How can I plan for unexpected expenses?",
    "What is the guaranteed interest rate for a bank loan?"
]

all_results = []

for question in test_questions:
    try:
        result = explain_financial_query(question)

        all_results.append({
            "Question": result.get("question", question),
            "Explanation": result.get("explanation", ""),
            "Evidence_IDs": ", ".join(
                result.get("evidence_ids", [])
            ),
            "Sources": ", ".join(
                result.get("sources", [])
            ),
            "Evidence_Status": result.get(
                "evidence_status", "Not verified"
            )
        })

    except Exception as e:
        all_results.append({
            "Question": question,
            "Explanation": f"Error: {str(e)}",
            "Evidence_IDs": "",
            "Sources": "",
            "Evidence_Status": "Failed"
        })

results_df = pd.DataFrame(all_results)

display(results_df)

results_df.to_csv(
    "evidence_based_financial_explanations.csv",
    index=False
)

print("\nTesting completed!")
print("Total questions:", len(results_df))
print(
    "Successful:",
    (results_df["Evidence_Status"] != "Failed").sum()
)
print("Report saved successfully!")


,Question,Explanation,Evidence_IDs,Sources,Evidence_Status
0,How can I improve my credit score?,Keeping credit utilization low is generally pr...,"E2, E3, E1","credit_utilization_guide, loan_eligibility_gui...",Evidence retrieved; verify relevance
1,What is credit utilization?,Keeping credit utilization low is generally pr...,"E2, E1","credit_utilization_guide, credit_score_guide",Evidence retrieved; verify relevance
2,Does a high credit score guarantee loan approval?,"Loan approval depends on lender requirements, ...","E3, E2, E1","loan_eligibility_guide, credit_utilization_gui...",Evidence retrieved; verify relevance
3,How do I calculate my debt-to-income ratio?,Debt-to-income ratio is calculated by dividing...,E4,debt_management_guide,Evidence retrieved; verify relevance
4,How can I plan for unexpected expenses?,"Create a monthly budget, track spending and pr...",E5,budgeting_guide,Evidence retrieved; verify relevance
5,What is the guaranteed interest rate for a ban...,"Loan approval depends on lender requirements, ...","E3, E1","loan_eligibility_guide, credit_score_guide",Evidence retrieved; verify relevance



Testing completed!
Total questions: 6
Successful: 6
Report saved successfully!


In [6]:

from google.colab import files

files.download("evidence_based_financial_explanations.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>